# Parkinson Model Training On Colab

Use this notebook to train the frozen DINO/DINOv2 baseline on a Colab GPU. Keep datasets and run outputs in Google Drive so they survive runtime resets.

Before running: select `Runtime > Change runtime type > T4 GPU` or better.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Clone Or Update The Repository

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/NIATS-UFU/parkinson-model-training.git"
REPO_DIR = Path("/content/parkinson-model-training")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(Path.cwd())

## 3. Install Dependencies

The project uses `uv`. This cell installs `uv`, creates a Python 3.13 environment, and installs the package dependencies.

In [ ]:
import os
import subprocess

subprocess.run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, check=True)
os.environ["PATH"] = f"{Path.home()}/.local/bin:" + os.environ["PATH"]
subprocess.run(["uv", "sync", "--all-groups", "--python", "3.13"], check=True)

## 4. Check GPU And PyTorch

If the CUDA check fails because Colab's driver does not match the locked PyTorch CUDA wheel, the cell tries a Colab-friendly CUDA 12.4 PyTorch wheel inside the project virtual environment.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=False)

check_cuda = [
    ".venv/bin/python",
    "-c",
    "import torch; print('torch', torch.__version__); print('cuda available', torch.cuda.is_available()); raise SystemExit(0 if torch.cuda.is_available() else 1)",
]
result = subprocess.run(check_cuda, check=False)

if result.returncode != 0:
    print("CUDA was not available in the project environment. Trying PyTorch CUDA 12.4 wheels for Colab...")
    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "--python",
            ".venv/bin/python",
            "--force-reinstall",
            "torch",
            "torchvision",
            "--index-url",
            "https://download.pytorch.org/whl/cu124",
        ],
        check=True,
    )
    subprocess.run(check_cuda, check=True)

## 5. Configure Your Dataset And Run

Edit only this cell for your first run. `DATASET_KIND` can be `hemato`, `niats-folador`, or `image-tree`.

`hemato` is the frozen key for the **Moura** cohort (the NIATS-UFU clinical dataset). It expects a `spiral-cropper` tree with `5_final/` crop folders.

In [ ]:
from pathlib import Path

DATASET_KIND = "hemato"
DATA_ROOT = Path("/content/drive/MyDrive/parkinson-data/hemato")
MANIFEST = Path("data/manifests/hemato_colab.csv")
OUTPUT_DIR = Path("/content/drive/MyDrive/parkinson-runs/dinov2-hemato")

MODEL_NAME = "vit_small_patch14_dinov2"
IMAGE_SIZE = 518
EPOCHS = 20
BATCH_SIZE = 16
SEED = 42

print(f"Dataset root: {DATA_ROOT}")
print(f"Manifest: {MANIFEST}")
print(f"Output dir: {OUTPUT_DIR}")

## 6. Create And Validate Manifest

In [ ]:
import subprocess

MANIFEST.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(["uv", "run", "main-cli", "create-manifest", DATASET_KIND, str(DATA_ROOT), str(MANIFEST)], check=True)
subprocess.run(["uv", "run", "main-cli", "validate-manifest", str(MANIFEST)], check=True)
subprocess.run(
    [
        "uv",
        "run",
        "main-cli",
        "split-summary",
        str(MANIFEST),
        "--val-fraction",
        "0.15",
        "--test-fraction",
        "0.15",
        "--seed",
        str(SEED),
    ],
    check=True,
)

## 7. Train

In [ ]:
import subprocess

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        "uv",
        "run",
        "main-cli",
        "train",
        str(MANIFEST),
        str(OUTPUT_DIR),
        "--model-name",
        MODEL_NAME,
        "--image-size",
        str(IMAGE_SIZE),
        "--epochs",
        str(EPOCHS),
        "--batch-size",
        str(BATCH_SIZE),
        "--device",
        "cuda",
        "--seed",
        str(SEED),
    ],
    check=True,
)

## 8. Inspect Outputs

In [ ]:
import json

from IPython.display import SVG, display

summary_path = OUTPUT_DIR / "run_summary.json"
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))

for plot_name in ["split_counts.svg", "validation_history.svg", "test_metrics.svg"]:
    plot_path = OUTPUT_DIR / plot_name
    if plot_path.exists():
        print(plot_path)
        display(SVG(filename=str(plot_path)))

## If Training Runs Out Of Memory

Lower `BATCH_SIZE` from `16` to `8` or `4`, then rerun the training cell. Keep `IMAGE_SIZE = 518` for `vit_small_patch14_dinov2`.